# Vendor Master Data Quality

Initial scope: SAP-style vendor master data.

Phase 1 focus:
- General vendor data
- Data profiling
- Deterministic Data Quality rules
- Duplicate and similarity analysis later

In [180]:
import pandas as pd
vendor_columns = [
    "LIFNR",
    "LAND1",
    "NAME1",
    "NAME2",
    "ORT01",
    "PSTLZ",
    "STRAS",
    "TELF1",
    "STCD1",
    "SPERR",
    "LOEVM"
]

len(vendor_columns)

11

In [181]:
vendor_data = {
    "LIFNR": [
        "100001", "100002", "100003", "100004", "100005",
        "100006", "100007", "100008", "100009", "100010",
        "100011", "100012"
    ],

"KTOKK": [
    "ZSUP", "ZSUP", "ZSUP", "ZSUP", "ZSUP",
    "ZSUP", "ZSUP", "ZSER", "ZSUP", "ZSER",
    "ZSUP", "ZSUP"
],

    "LAND1": [
        "AE", "AE", "AE", "OM", "AE",
        "AE", "AE", None, "AE", "OM",
        "AE", "AE"
    ],

    "NAME1": [
        "Gulf Trading LLC",
        "Gulf Trading L.L.C.",
        "Al Noor Services",
        "Muscat Industrial Co",
        None,
        "Desert Engineering LLC",
        "DESERT ENGINEERING L.L.C test",
        "Emirates Technical Services NA",
        "Al Falah NA Trading",
        "Oman Equipment Services do not use",
        "Future Tech Solutions",
        "Future Tech Solution LLC"
    ],

    "NAME2": [
        None,
        None,
        "Maintenance Division",
        None,
        None,
        None,
        None,
        None,
        "Abu Dhabi Branch",
        None,
        None,
        None
    ],

    "ORT01": [
        "Abu Dhabi",
        "Abu Dhabi",
        "Dubai",
        "Muscat",
        "Dubai",
        "Abu Dhabi",
        "ABU DHABI",
        "Sharjah",
        "AbuDhabi",
        "Muscat",
        "Dubai",
        "Dubai"
    ],

    "PSTLZ": [
        "12345",
        "12345",
        None,
        "112",
        "00000",
        "45678",
        "45678",
        "54321",
        "12A45",
        "113",
        "67890",
        "67890"
    ],

    "STRAS": [
        "Mussafah Industrial Area",
        "Mussafah Ind. Area",
        "Sheikh Zayed Road",
        "Ruwi Industrial Estate",
        None,
        "Street 10, Mussafah",
        "Street 10 Mussafah",
        "Industrial Area 4",
        "Hamdan Street",
        "Ghala Industrial Area",
        "Business Bay Tower 2",
        "Business N/A, Tower 2"
    ],

    "TELF1": [
        "+971501234567",
        "0501234567",
        "+971 55 222 3344",
        "+96899112233",
        None,
        "02-5556677",
        "+97125556677",
        "12345",
        "+971501111222",
        "+968 24 567890",
        "+971504445555",
        "0504445555"
    ],

    "STCD1": [
        "100234567890003",
        "100234567890003",
        "100345678900003",
        "OM1234567",
        None,
        "100456789010003",
        "100456789010003",
        "ABC123",
        "100567890120003",
        "OM9876543",
        "100678901230003",
        "100678901230003"
    ],

    "SPERR": [
        "", "", "", "", "",
        "", "X", "", "", "",
        "", ""
    ],

    "LOEVM": [
        "", "", "", "", "",
        "", "", "", "X", "",
        "", ""
    ]
}


raw_data = pd.DataFrame(vendor_data)
#print(vendor_raw.info())
data = raw_data.copy()

rule_catalog=pd.read_excel(r'/Users/vivek/Documents/Data Science Projects/data-quality-solution/DQ_Rule_Catalog.xlsx')
dq_results = pd.DataFrame( columns=["WORKSTREAM","OBJECT","OBJECT_KEY","OBJECT_KEY_VALUE","RULE_ID","DQ_DIMENSION","FIELD","ISSUE"])


In [182]:

def add_dq_results(data,dq_results, condition, workstream, object , key, field, rule_id, dq_dimension, issue):
    rule_results = data[condition][[key]].copy()
    #print(len(rule_results))
    if(len(rule_results)>0):
        rule_results.rename(columns={key: "OBJECT_KEY_VALUE"}, inplace=True)
        rule_results["WORKSTREAM"]=workstream
        rule_results["OBJECT"]=object
        rule_results["OBJECT_KEY"]=key
        rule_results["RULE_ID"]=rule_id
        rule_results["DQ_DIMENSION"] = dq_dimension
        rule_results["FIELD"] = field
        rule_results["ISSUE"]= issue
        dq_results = pd.concat([dq_results, rule_results], ignore_index = True)

    #print(rule_results)
    
    dq_results = dq_results.drop_duplicates(
            subset=[
                "WORKSTREAM",
                "OBJECT",
                "OBJECT_KEY_VALUE",
                "RULE_ID"
            ],
            keep="last"
        )

    return dq_results
    #print(dq_results)


In [183]:
# Log for Standard Rule Category 
standard_missing = rule_catalog[ (rule_catalog["RULE_CLASS"]=="STANDARD") & (rule_catalog["RULE_TYPE"]=="MISSING")  ]

#standard_missing
for row in standard_missing.itertuples():
    condition = (data[row.FIELD].isna())  | (data[row.FIELD].str.strip()=='')
    dq_results = add_dq_results(data,dq_results,condition,row.WORKSTREAM,row.OBJECT,row.KEY,row.FIELD,row.RULE_ID,row.DQ_DIMENSION,row.ISSUE)
    
#dq_results


In [184]:
# Logic for PLACEHOLDER_OR_SUSPICIOUS_TEXT category
import re

placeholder_values = [
    "TEST",
    "DUMMY",
    "XXX",
    "N/A",
    "NA",
    "UNKNOWN",
    "TBD",
    "TEMP",
    "SAMPLE",
    "DELETE",
    "DELETED",
    "REMOVE",
    "REMOVED",
    "NOT REQUIRED",
    "NOT APPLICABLE",
    "NOT NEEDED",
    "DO NOT USE",
    "DONT USE",
    "OBSOLETE"
]


pattern = r"\b(" + "|".join(
    re.escape(value) for value in placeholder_values
) + r")\b"

data[data["STRAS"].str.contains(pattern,case=False,na=False,regex=True)]

SUSPICIOUS_TEXT = rule_catalog[ (rule_catalog["RULE_CLASS"]=="CUSTOM") & (rule_catalog["RULE_TYPE"]=="PLACEHOLDER_OR_SUSPICIOUS_TEXT")  ]

# PLACEHOLDER_OR_SUSPICIOUS_TEXT
for row in SUSPICIOUS_TEXT.itertuples():
    condition = data[row.FIELD].str.contains(pattern,case=False,na=False,regex=True)
    dq_results = add_dq_results(data,dq_results,condition,row.WORKSTREAM,row.OBJECT,row.KEY,row.FIELD,row.RULE_ID,row.DQ_DIMENSION,row.ISSUE)



/var/folders/5b/b99k41_n0sdg438t9w08svk00000gn/T/ipykernel_2145/1478555724.py:31: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  data[data["STRAS"].str.contains(pattern,case=False,na=False,regex=True)]
/var/folders/5b/b99k41_n0sdg438t9w08svk00000gn/T/ipykernel_2145/1478555724.py:37: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  condition = data[row.FIELD].str.contains(pattern,case=False,na=False,regex=True)


In [185]:
# Logic for PHONE_VALIDATION 

PHONE_VALIDATION = rule_catalog[ (rule_catalog["RULE_CLASS"]=="CUSTOM") & (rule_catalog["RULE_TYPE"]=="PHONE_VALIDATION_BY_COUNTRY")  ]

# PHONE_VALIDATION
for row in PHONE_VALIDATION.itertuples():
    condition = ~data[row.FIELD].str.fullmatch(r"\+?\s?[0-9][0-9\s\-()]+",na=False)
    dq_results = add_dq_results(data,dq_results,condition,row.WORKSTREAM,row.OBJECT,row.KEY,row.FIELD,row.RULE_ID,row.DQ_DIMENSION,row.ISSUE)


In [186]:
# Logic for POSTAL CODE VALIDATION 

POSTAL_CODE_VALIDATION = rule_catalog[ (rule_catalog["RULE_CLASS"]=="CUSTOM") & (rule_catalog["RULE_TYPE"]=="POSTAL_FORMAT_BY_COUNTRY")  ]

# POSTAL_CODE_VALIDATION
for row in POSTAL_CODE_VALIDATION.itertuples():
    condition = ~data[row.FIELD].str.fullmatch(r"[a-zA-Z0-9\s-]+",na=False)
    dq_results = add_dq_results(data,dq_results,condition,row.WORKSTREAM,row.OBJECT,row.KEY,row.FIELD,row.RULE_ID,row.DQ_DIMENSION,row.ISSUE)


In [187]:
# Logic for TAX CODE VALIDATION 

TAX_CODE_VALIDATION = rule_catalog[ (rule_catalog["RULE_CLASS"]=="CUSTOM") & (rule_catalog["RULE_TYPE"]=="TAX_FORMAT_BY_COUNTRY")  ]

country_list = data["LAND1"].unique()
#print(country_list)


for country in country_list:
    if country=='AE':
        pattern = r"[0-9]{15}"
    elif country=='OM':
        pattern = r"[A-Za-z0-9]{12}"
    country_data = data[data["LAND1"]==country]
    

# TAX_CODE_VALIDATION
    for row in TAX_CODE_VALIDATION.itertuples():
        condition = ~country_data[row.FIELD].str.fullmatch(pattern,na=False)
        dq_results = add_dq_results(country_data,dq_results,condition,row.WORKSTREAM,row.OBJECT,row.KEY,row.FIELD,row.RULE_ID,row.DQ_DIMENSION,row.ISSUE)


In [188]:
#dq_results
#re.fullmatch(r"[A-Za-z0-9]{12}",'OM1234567123')
